<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 40px; border-radius: 12px; color: white; margin-bottom: 20px;">
  <h1 style="margin: 0; font-size: 2em;">🔧 Lab 03: Add Financial Tools to Your MAF Agent</h1>
  <p style="margin: 8px 0 0 0; font-size: 1.15em; opacity: 0.92;">Zava Bank Workshop — Hosted Agents with Microsoft Agent Framework</p>
</div>

## What We Learn

This lab adds **portfolio, risk, and market-data tools** to our MAF agent using Python's `Annotated` type-hint pattern.

Key difference from Path A: MAF tools are plain Python functions. The framework **auto-generates the JSON schema** from your type hints — no `FunctionTool` descriptor or manual schema definition required. Tools execute **in-process**, so there's no HTTP round-trip and debugging is straightforward.

## The Zava Bank Story

In **Path A (Prompt Agents)**, we defined tools with JSON schemas and registered them server-side. The platform executed the tools on our behalf, and the agent only saw the serialized input/output.

In **Path B (MAF)**, tools are Python functions decorated with `Annotated` type hints. The framework inspects those hints at init time and builds the tool schema automatically. This is:

- **Faster to develop** — write a function, add type hints, done.
- **Easier to test** — call the function directly in a cell.
- **Easier to debug** — breakpoints and print statements work as expected.

Both paths produce equivalent results for the end user; the difference is developer experience.

---
## 1 · Setup

In [ ]:
import os, json, pathlib
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# Verify Azure OpenAI config
for key in ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_DEPLOYMENT"]:
    assert os.getenv(key), f"Missing env var: {key}"
print("✅ Environment loaded")

In [ ]:
# Resolve data path
DATA_DIR = pathlib.Path("../../data/zava-bank")
assert DATA_DIR.exists(), f"Data directory not found at {DATA_DIR.resolve()}"

portfolios_df = pd.read_csv(DATA_DIR / "client_portfolios.csv")
risk_df       = pd.read_csv(DATA_DIR / "risk_metrics.csv")
market_df     = pd.read_csv(DATA_DIR / "market_data.csv")

print(f"📊 Loaded — portfolios: {len(portfolios_df)} rows | risk: {len(risk_df)} rows | market: {len(market_df)} rows")

---
## 2 · Define Tool Functions with `Annotated` Type Hints

Each parameter uses `Annotated[type, "description"]`. MAF reads these at init and builds the OpenAI-compatible tool schema automatically.

In [ ]:
from typing import Annotated


def search_portfolio(
    client_name: Annotated[str, "Client name to search for"] = None,
    ticker: Annotated[str, "Stock ticker symbol"] = None,
    sector: Annotated[str, "Sector to filter by"] = None,
) -> Annotated[str, "JSON array of matching portfolio positions"]:
    """Search Zava Bank client portfolio holdings."""
    results = portfolios_df.copy()
    if client_name:
        results = results[results["client_name"].str.lower().str.contains(client_name.lower())]
    if ticker:
        results = results[results["ticker"].str.upper() == ticker.upper()]
    if sector:
        results = results[results["sector"].str.lower().str.contains(sector.lower())]
    if results.empty:
        return json.dumps({"message": "No holdings found."})
    return results.to_json(orient="records")


def assess_risk(
    client_name: Annotated[str, "Client name"] = None,
    risk_tolerance: Annotated[str, "Filter by risk tolerance level"] = None,
) -> Annotated[str, "JSON array of risk metrics"]:
    """Get risk assessment for a client portfolio."""
    results = risk_df.copy()
    if client_name:
        results = results[results["client_name"].str.lower().str.contains(client_name.lower())]
    if risk_tolerance:
        results = results[results["risk_tolerance"].str.lower() == risk_tolerance.lower()]
    if results.empty:
        return json.dumps({"message": "No risk data found."})
    return results.to_json(orient="records")


def get_market_data(
    category: Annotated[str, "Category: Index, Sector Performance, or Economic Indicator"] = None,
    name: Annotated[str, "Indicator name to search for"] = None,
) -> Annotated[str, "JSON array of market data"]:
    """Get current market data, sector performance, and economic indicators."""
    results = market_df.copy()
    if category:
        results = results[results["category"].str.lower().str.contains(category.lower())]
    if name:
        results = results[results["name"].str.lower().str.contains(name.lower())]
    if results.empty:
        return json.dumps({"message": "No market data found."})
    return results.to_json(orient="records")


print("✅ Tool functions defined: search_portfolio, assess_risk, get_market_data")

### Quick sanity check — call a tool directly

Because these are plain functions, we can test them without the agent.

In [ ]:
# Direct function call — no agent involved
sample = json.loads(search_portfolio(client_name="Margaret Chen"))
print(f"Margaret Chen holds {len(sample)} positions")
pd.DataFrame(sample)[["ticker", "company", "sector", "market_value"]]

---
## 3 · Build the Agent with Tools

We wire the three tool functions into the MAF agent. The framework handles schema generation, tool-call parsing, function dispatch, and result injection back into the conversation.

In [ ]:
from microsoft.agents.builder import BaseAgent, AgentContext


class ZavaBankAdvisorWithTools(BaseAgent):
    """Zava Bank advisor agent equipped with financial data tools."""

    tools = [search_portfolio, assess_risk, get_market_data]

    SYSTEM_PROMPT = """You are the Zava Bank Client Advisor with access to real financial data.
Use your tools to answer questions:
- search_portfolio: Look up client holdings
- assess_risk: Get risk metrics
- get_market_data: Get market conditions
Always ground your responses in the data returned by your tools.
When presenting numbers use clear formatting. If a question is ambiguous, state your assumptions."""

    async def on_message(self, context: AgentContext, message: str) -> str:
        response = await context.invoke_model(
            system_prompt=self.SYSTEM_PROMPT,
            user_message=message,
            tools=self.tools,
        )
        return response


advisor = ZavaBankAdvisorWithTools()
print("✅ ZavaBankAdvisorWithTools created with", len(advisor.tools), "tools")

---
## 4 · Test: Portfolio + Risk Query

This query should trigger both `search_portfolio` and `assess_risk` tool calls.

In [ ]:
import asyncio

query_1 = "What does Priya Kapoor's portfolio look like and what are her risk metrics?"

response_1 = await advisor.on_message(
    context=advisor.create_context(),
    message=query_1,
)

print(f"🗣️ Query: {query_1}\n")
print(f"🤖 Response:\n{response_1}")

---
## 5 · Test: Market Overlay Query

This query should trigger `search_portfolio` (to find Apex Capital's sectors) and `get_market_data` (to pull sector performance).

In [ ]:
query_2 = "Give me a market overview focused on the sectors where Apex Capital has exposure."

response_2 = await advisor.on_message(
    context=advisor.create_context(),
    message=query_2,
)

print(f"🗣️ Query: {query_2}\n")
print(f"🤖 Response:\n{response_2}")

---
## 6 · Path A vs Path B: How Tools Compare

| Dimension | Path A (Prompt Agents) | Path B (MAF) |
|---|---|---|
| **Definition** | JSON schema + `FunctionTool` descriptor | Python function with `Annotated` type hints |
| **Execution** | Server-side; platform calls your endpoint | In-process; framework calls the function directly |
| **Schema** | Hand-authored or generated offline | Auto-generated at init from type hints |
| **Testing** | Requires mock server or integration test | Call the function in a cell |
| **Debugging** | Logs + request/response traces | Breakpoints, print statements, step-through |
| **Output** | Equivalent — same quality answers | Equivalent — same quality answers |

Both approaches produce the same result for the end user. The MAF path reduces boilerplate and keeps the tool logic co-located with the agent code, which is a better fit for rapid prototyping and workshops.

---
## 7 · Cleanup

In [ ]:
del advisor
print("🧹 Cleanup complete")

---
<div style="background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 20px 30px; border-radius: 10px; color: white;">
  <h2 style="margin: 0 0 8px 0;">✅ Lab 03 Summary</h2>
  <ul style="margin: 0; padding-left: 20px; line-height: 1.8;">
    <li>MAF's <code>Annotated</code> type hints eliminate hand-written JSON schema definitions.</li>
    <li>Tools execute <strong>in the same Python process</strong> as the agent — no HTTP boundary.</li>
    <li>You can call and test tool functions directly, outside the agent loop.</li>
    <li>The framework auto-generates the OpenAI-compatible tool schema from your function signatures.</li>
  </ul>
  <p style="margin: 12px 0 0 0; opacity: 0.85;">Next → <strong>Lab 04</strong>: Trace your MAF agent to see how in-process tool execution shows up in spans.</p>
</div>